<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Proyección de la ganancia exportadora de Colombia

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `pickle`: para recuperar el modelo estimado.

In [1]:
import pandas as pd
import numpy as np
import pickle

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp, processed, model

### Carga del modelo estimado

Recupera el modelo lineal generalizado ajustado con la experiencia de terceros países, que es el
que se aplica al caso colombiano.

In [3]:
# Cargar el modelo desde el archivo
with open(model / 'glm_model.pkl', 'rb') as f:
    model_glm = pickle.load(f)

# Usar el modelo cargado
print(model_glm .summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                Delta_x   No. Observations:               353741
Model:                            GLM   Df Residuals:                   353694
Model Family:                 Poisson   Df Model:                           46
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.3501e+10
Date:                Sun, 20 Sep 2026   Deviance:                   2.6998e+10
Time:                        07:42:26   Pearson chi2:                 4.62e+11
No. Iterations:                    11   Pseudo R-squ. (CS):              1.000
Covariance Type:                  HC3                                         
                                              coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

### Variables de la proyección

Define el conjunto de regresores que debe reconstruirse para Colombia, en el mismo orden con
que se estimó el modelo.

In [4]:
list_vars = ['const', 'log_Delta_m', 'share_start', 
             'log_duracion', 'log_duracion2',
            'log_gdp_o', 'log_gdp_o2',   
            'log_gdp_d', 'log_gdp_d2',
            'cultura_similaridad', 'log_distancia', 'subregion_o_Eastern Asia',
            'subregion_o_Eastern Europe', 'subregion_o_Latin America and Caribbean',
            'subregion_o_Northern America', 'subregion_o_Northern Europe',
            'subregion_o_Other Asia and Oceania', 'subregion_o_South-Eastern Asia',
            'subregion_o_Southern Europe', 'subregion_o_Western Asia',
            'subregion_o_Western Europe', 'subregion_d_Eastern Asia',
            'subregion_d_Eastern Europe', 'subregion_d_Latin America and Caribbean',
            'subregion_d_Northern America', 'subregion_d_Northern Europe',
            'subregion_d_Other Asia and Oceania', 'subregion_d_South-Eastern Asia',
            'subregion_d_Southern Europe', 'subregion_d_Western Asia',
            'subregion_d_Western Europe', 'industria_1 - 108', 'industria_1 - 141',
            'industria_1 - Otros', 'industria_10 - Otros', 'industria_2 - 201',
            'industria_2 - 202', 'industria_2 - 222', 'industria_2 - 281',
            'industria_2 - 293', 'industria_2 - Otros', 'industria_24 - Otros',
            'industria_25 - Otros', 'industria_26 - Otros', 'industria_27 - Otros',
            'industria_3 - Otros', 'industria_Otras industrias']

### Panel de comercio y demanda proyectada

Carga el panel de participaciones y la proyección de importaciones por producto y destino.

In [5]:
df_trade = pd.read_parquet(raw / 'modelo' / 'df_trade_share.parquet')
df_trade.drop(columns=['region_o','region_d','ingreso_o','ingreso_d','food_index','tipo_oportunidad'], inplace=True)
df_trade

,anio,hs6_1996,nombre_o,o_iso3,nombre_d,d_iso3,valor_miles_usd,deflactor_2019,distancia,cultura_similaridad,total_prod_year,share
0,2004,620990,Thailand,THA,Angola,AGO,83.69,1.31,"9,947.91",43.80,635.11,0.13
1,2004,620990,Vietnam,VNM,Angola,AGO,77.22,1.31,"10,627.53",48.60,635.11,0.12
2,2004,621132,Brazil,BRA,Angola,AGO,212.80,1.31,"6,663.72",62.80,212.80,1.00
3,2004,621133,Portugal,PRT,Angola,AGO,581.76,1.31,"5,771.92",51.70,581.76,1.00
4,2004,621142,Brazil,BRA,Angola,AGO,67.07,1.31,"6,663.72",62.80,67.07,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...
15555429,1997,730890,Zimbabwe,ZWE,Zambia,ZMB,87.51,1.50,398.72,82.80,"4,305.55",0.02
15555430,1997,731512,South Africa,ZAF,Zambia,ZMB,118.46,1.50,"1,538.04",65.90,118.46,1.00
15555431,1997,731829,United Arab Emirates,ARE,Zambia,ZMB,"11,900.05",1.50,"5,261.02",37.90,"12,481.80",0.95
15555432,1997,731829,Finland,FIN,Zambia,ZMB,280.45,1.50,"8,410.32",39.70,"12,481.80",0.02


In [6]:
df_delta_m = pd.read_parquet(temp / 'exportaciones' / 'df_m_forecast_6_8_25.parquet')
# df_delta_m.query("anio_pred == 2023") ## Suponemos que si se predijo que hay un 2023 es porque en el 2023 no demandaba y no se hace la proyección con esos datos
df_delta_m.query("anio_pred == 2035", inplace=True)
df_delta_m.drop(columns=['anio_pred'], inplace=True)
df_delta_m.reset_index(drop=True, inplace=True)
df_delta_m['m_hat'] = np.where(df_delta_m['m_hat'] < 0, 0, df_delta_m['m_hat']) # Si la predicción es negativa, se pone en 0
df_delta_m

,d_iso3,hs6_1996,m_hat
0,ESP,620990,"1,993.21"
1,ESP,621132,"38,623.94"
2,ESP,621133,"56,419.24"
3,ESP,621142,"168,370.46"
4,ESP,621143,"232,379.26"
...,...,...,...
44858,ABW,071390,0.00
44859,ABW,060491,991.79
44860,ABW,070390,217.69
44861,ABW,960891,0.00


### Situación exportadora inicial de Colombia

Toma como punto de partida las exportaciones colombianas del último año observado y la demanda
de cada mercado de destino.

In [7]:
df_col = df_trade.query("anio == 2023 and o_iso3 == 'COL'").reset_index(drop=True)
df_col.drop(columns=[ 'valor_miles_usd', 'deflactor_2019','total_prod_year', 'nombre_d', 'nombre_o','distancia','cultura_similaridad'], inplace=True)
df_col.rename(columns={'share':'share_start'}, inplace=True)
df_col

,anio,hs6_1996,o_iso3,d_iso3,share_start
0,2023,020130,COL,ARE,0.01
1,2023,870899,COL,ABW,0.02
2,2023,190530,COL,AGO,0.01
3,2023,701092,COL,ABW,0.38
4,2023,847989,COL,AGO,0.00
...,...,...,...,...,...
4328,2023,170490,COL,ZAR,0.08
4329,2023,110720,COL,ZAF,0.03
4330,2023,950390,COL,ZAF,0.00
4331,2023,392112,COL,ZAF,0.01


In [8]:
df_demanda_inicial = df_trade[['anio','d_iso3', 'hs6_1996','total_prod_year']].query("anio == 2023 and d_iso3 != 'COL'").drop_duplicates().reset_index(drop=True).rename(columns={'total_prod_year':'m_start'})
df_demanda_inicial

,anio,d_iso3,hs6_1996,m_start
0,2023,AGO,610910,"6,171.72"
1,2023,AGO,610990,"1,906.39"
2,2023,AGO,611020,586.10
3,2023,AGO,611030,309.12
4,2023,AGO,611090,"1,511.32"
...,...,...,...,...
76781,2023,ZWE,940340,261.05
76782,2023,ZWE,940350,394.74
76783,2023,ZWE,940360,"1,998.58"
76784,2023,ZWE,940390,84.88


In [9]:
df_relacion_comercial = df_demanda_inicial.merge(df_col, on=['d_iso3','hs6_1996', 'anio'], how='left')
df_relacion_comercial['o_iso3'] = 'COL'
df_relacion_comercial['share_start'] = df_relacion_comercial['share_start'].fillna(0)
df_relacion_comercial = df_relacion_comercial.merge(df_delta_m, on=['d_iso3','hs6_1996'], how='left').dropna().drop(columns=['anio']).reset_index(drop=True)
df_relacion_comercial['Delta_m'] = df_relacion_comercial['m_hat'] - df_relacion_comercial['m_start']
df_relacion_comercial['log_Delta_m'] = np.sign(df_relacion_comercial['Delta_m']) * np.log1p(np.abs(df_relacion_comercial['Delta_m']))
df_relacion_comercial = df_relacion_comercial[['d_iso3','o_iso3', 'hs6_1996', 'log_Delta_m', 'share_start']]
df_relacion_comercial

,d_iso3,o_iso3,hs6_1996,log_Delta_m,share_start
0,ABW,COL,847990,5.44,0.00
1,ABW,COL,848350,-3.81,0.00
2,ABW,COL,848390,-4.06,0.00
3,ABW,COL,848590,-3.87,0.00
4,ABW,COL,850421,-3.91,0.00
...,...,...,...,...,...
37539,VNM,COL,940592,8.89,0.00
37540,VNM,COL,940600,-7.83,0.00
37541,VNM,COL,950390,11.10,0.00
37542,VNM,COL,960200,-7.18,0.00


### Controles y producto interno bruto

Adjunta la distancia, la similitud cultural y el producto interno bruto per cápita de origen y
destino, en precios constantes.

In [10]:
df_controles_i = pd.read_parquet(temp / 'exportaciones' / 'df_para_estimacion.parquet') # regiones, INDUSTRIA 

df_regiones = pd.concat([
        df_controles_i[['o_iso3', 'subregion_o']].drop_duplicates().rename(columns={'o_iso3':'iso3', 'subregion_o':'subregion'}).reset_index(drop=True),
        df_controles_i[['d_iso3', 'subregion_d']].drop_duplicates().rename(columns={'d_iso3':'iso3', 'subregion_d':'subregion'}).reset_index(drop=True)
], ignore_index=True).drop_duplicates().reset_index(drop=True)

df_industrias = df_controles_i[['hs6_1996', 'industria']].drop_duplicates().reset_index(drop=True)

In [11]:
df_gdp = (
    pd.read_excel(raw / 'exportaciones' / 'gdp' / 'Emparejamiento_IMF_-_ISO.xlsx')
    .merge(
        pd.read_excel(raw / 'exportaciones' / 'gdp' / 'imf-dm-export-20250806.xls'),
        right_on="GDP, current prices (Billions of U.S. dollars)",
        left_on="nombre_imf")
    .drop(columns=["GDP, current prices (Billions of U.S. dollars)"])
    .melt(
        id_vars=["nombre_imf", "iso3"],   # Columnas que se mantienen fijas
        var_name="anio",                  # Nueva columna con los años
        value_name="gdp")                # Nombre de la columna con los valores
    .query("anio == 2023")
    .reset_index(drop=True)
    .drop(columns=["nombre_imf"])
    .merge(pd.read_parquet(temp / 'exportaciones' / 'deflactor.parquet'), on=["anio"], how="left")
)
df_gdp['gdp_deflactado'] = df_gdp['gdp'] * df_gdp['deflactor_2019'] * 1_000_000
df_gdp.drop(columns=['gdp','deflactor_2019'], inplace=True)
df_gdp.rename(columns={"gdp_deflactado": "gdp"}, inplace=True)
df_gdp

,iso3,anio,gdp
0,AFG,2023,"14,667,627.76"
1,ALB,2023,"19,889,058.33"
2,DZA,2023,"210,718,739.31"
3,AND,2023,"3,219,598.72"
4,AGO,2023,"93,342,850.98"
...,...,...,...
188,VEN,2023,"87,060,976.78"
189,VNM,2023,"368,228,209.77"
190,YEM,2023,"16,507,884.40"
191,ZMB,2023,"23,452,216.98"


In [12]:
df = (
    df_relacion_comercial
    .merge(df_regiones, left_on='o_iso3', right_on='iso3', how='left').rename(columns={'subregion':'subregion_o'}).drop(columns=['iso3'])
    .merge(df_regiones, left_on='d_iso3', right_on='iso3', how='left').rename(columns={'subregion':'subregion_d'}).drop(columns=['iso3'])
    .merge(df_industrias, on='hs6_1996', how='left')
    .merge(df_trade[['o_iso3','d_iso3','distancia','cultura_similaridad']].drop_duplicates().reset_index(drop=True), on=['o_iso3','d_iso3'], how='left')
    .merge(df_gdp, left_on=['o_iso3'], right_on=['iso3'], how='left').drop(columns=['iso3'])
    .merge(df_gdp, left_on=['d_iso3'], right_on=['iso3'], how='left',suffixes=('_o', '_d')).drop(columns=['iso3'])
    .fillna({'industria':'Otras industrias'})
    .dropna() # Estoy eliminando las exportaciones a Barbados e Islas Caimán
    .reset_index(drop=True)
)
# df.isnull().sum().sum()
df

,d_iso3,o_iso3,hs6_1996,log_Delta_m,share_start,subregion_o,subregion_d,industria,distancia,cultura_similaridad,anio_o,gdp_o,anio_d,gdp_d
0,ABW,COL,847990,5.44,0.00,Latin America and Caribbean,Latin America and Caribbean,2 - Otros,974.83,68.06,2023,"311,493,199.70",2023,"3,103,094.49"
1,ABW,COL,848350,-3.81,0.00,Latin America and Caribbean,Latin America and Caribbean,2 - 281,974.83,68.06,2023,"311,493,199.70",2023,"3,103,094.49"
2,ABW,COL,848390,-4.06,0.00,Latin America and Caribbean,Latin America and Caribbean,2 - 281,974.83,68.06,2023,"311,493,199.70",2023,"3,103,094.49"
3,ABW,COL,848590,-3.87,0.00,Latin America and Caribbean,Latin America and Caribbean,2 - 281,974.83,68.06,2023,"311,493,199.70",2023,"3,103,094.49"
4,ABW,COL,850421,-3.91,0.00,Latin America and Caribbean,Latin America and Caribbean,27 - Otros,974.83,68.06,2023,"311,493,199.70",2023,"3,103,094.49"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36846,VNM,COL,940592,8.89,0.00,Latin America and Caribbean,South-Eastern Asia,2 - 222,"17,156.13",53.20,2023,"311,493,199.70",2023,"368,228,209.77"
36847,VNM,COL,940600,-7.83,0.00,Latin America and Caribbean,South-Eastern Asia,1 - Otros,"17,156.13",53.20,2023,"311,493,199.70",2023,"368,228,209.77"
36848,VNM,COL,950390,11.10,0.00,Latin America and Caribbean,South-Eastern Asia,3 - Otros,"17,156.13",53.20,2023,"311,493,199.70",2023,"368,228,209.77"
36849,VNM,COL,960200,-7.18,0.00,Latin America and Caribbean,South-Eastern Asia,3 - Otros,"17,156.13",53.20,2023,"311,493,199.70",2023,"368,228,209.77"


### Matriz de diseño fuera de muestra

Construye las variables indicadoras de subregión e industria y las transformaciones
logarítmicas, en el mismo orden que la matriz con la que se estimó el modelo.

In [13]:
for var in ['subregion_d','subregion_o','industria']:
    # Obtener los elementos únicos de la columna 'region_d'
    unique_regions = df[var].unique()
    # Por cada elemento único, crear una columna dummy: True si la fila tiene ese valor
    for region in unique_regions:
        #cleaned = clean_str(region)
        col_name = f"{var}_{region}"
        df[col_name] = df[var] == region

In [14]:
df[['subregion_o_Eastern Asia', 'subregion_o_Eastern Europe', 'subregion_o_Northern America', 'subregion_o_Northern Europe', 'subregion_o_Other Asia and Oceania', 'subregion_o_South-Eastern Asia', 'subregion_o_Southern Europe', 'subregion_o_Western Asia', 'subregion_o_Western Europe', 'subregion_d_Western Asia']] = False
df

,d_iso3,o_iso3,hs6_1996,log_Delta_m,share_start,subregion_o,subregion_d,industria,distancia,cultura_similaridad,anio_o,gdp_o,anio_d,gdp_d,subregion_d_Latin America and Caribbean,subregion_d_Other Asia and Oceania,subregion_d_Western Europe,subregion_d_Eastern Europe,subregion_d_Northern America,subregion_d_Eastern Asia,subregion_d_Northern Europe,subregion_d_Southern Europe,subregion_d_South-Eastern Asia,subregion_o_Latin America and Caribbean,industria_2 - Otros,...,industria_27 - Otros,industria_26 - Otros,industria_3 - Otros,industria_2 - 293,industria_Otras industrias,industria_24 - Otros,industria_01 - Otros,industria_10 - Otros,industria_1 - 141,industria_2 - 202,industria_25 - Otros,industria_2 - 201,industria_2 - 222,industria_1 - Otros,industria_1 - 108,subregion_o_Eastern Asia,subregion_o_Eastern Europe,subregion_o_Northern America,subregion_o_Northern Europe,subregion_o_Other Asia and Oceania,subregion_o_South-Eastern Asia,subregion_o_Southern Europe,subregion_o_Western Asia,subregion_o_Western Europe,subregion_d_Western Asia
0,ABW,COL,847990,5.44,0.00,Latin America and Caribbean,Latin America and Caribbean,2 - Otros,974.83,68.06,2023,"311,493,199.70",2023,"3,103,094.49",True,False,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,ABW,COL,848350,-3.81,0.00,Latin America and Caribbean,Latin America and Caribbean,2 - 281,974.83,68.06,2023,"311,493,199.70",2023,"3,103,094.49",True,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,ABW,COL,848390,-4.06,0.00,Latin America and Caribbean,Latin America and Caribbean,2 - 281,974.83,68.06,2023,"311,493,199.70",2023,"3,103,094.49",True,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,ABW,COL,848590,-3.87,0.00,Latin America and Caribbean,Latin America and Caribbean,2 - 281,974.83,68.06,2023,"311,493,199.70",2023,"3,103,094.49",True,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,ABW,COL,850421,-3.91,0.00,Latin America and Caribbean,Latin America and Caribbean,27 - Otros,974.83,68.06,2023,"311,493,199.70",2023,"3,103,094.49",True,False,False,False,False,False,False,False,False,True,False,...,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36846,VNM,COL,940592,8.89,0.00,Latin America and Caribbean,South-Eastern Asia,2 - 222,"17,156.13",53.20,2023,"311,493,199.70",2023,"368,228,209.77",False,False,False,False,False,False,False,False,True,True,False,...,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False
36847,VNM,COL,940600,-7.83,0.00,Latin America and Caribbean,South-Eastern Asia,1 - Otros,"17,156.13",53.20,2023,"311,493,199.70",2023,"368,228,209.77",False,False,False,False,False,False,False,False,True,True,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
36848,VNM,COL,950390,11.10,0.00,Latin America and Caribbean,South-Eastern Asia,3 - Otros,"17,156.13",53.20,2023,"311,493,199.70",2023,"368,228,209.77",False,False,False,False,Fals

In [15]:
df['log_distancia'] = np.log1p(df['distancia'])

df['log_gdp_o'] = np.log1p(df['gdp_o'])
df['log_gdp_d'] = np.log1p(df['gdp_d'])

df['log_gdp_o2'] = df['log_gdp_o'] ** 2
df['log_gdp_d2'] = df['log_gdp_d'] ** 2

df['Duración'] = 10

df['log_duracion'] = np.log1p(df['Duración'])
df['log_duracion2'] = df['log_duracion'] ** 2
df['const'] = 1

X_fuera_muestra = df[list_vars].astype(float)

### Ganancia exportadora proyectada

Aplica los coeficientes estimados a la matriz fuera de muestra y reporta el total proyectado.

In [16]:
f"${(model_glm.predict(X_fuera_muestra).sum() * 1_000).round():,.0f}"

'$726,489,591,074'

### Exportación de resultados

Guarda la matriz fuera de muestra y las predicciones por producto, origen y destino.

In [17]:
(temp / 'exportaciones').mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)

pd.concat([df[['d_iso3', 'o_iso3', 'hs6_1996']], X_fuera_muestra], axis=1).to_parquet(temp / 'exportaciones' / 'X_fuera_muestra.parquet')

In [18]:
(temp / 'exportaciones').mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)

pd.concat([df[['d_iso3', 'o_iso3', 'hs6_1996']], X_fuera_muestra, model_glm.predict(X_fuera_muestra)], axis=1).rename(columns={0:'Prediction'}).to_parquet(processed / 'predicciones_proyeccion.parquet')

### Base combinada para la matriz insumo-producto

Arma la tabla que consume el cálculo de efectos sobre el producto interno bruto.

In [19]:
# CIERRE DE CADENA: df_hat_complex.parquet se consumia en la matriz insumo-producto pero
# ningun notebook lo producia. Se verifico que su columna Delta_x_hat es exactamente igual a
# la columna Prediction de predicciones_proyeccion, de modo que se arma aqui a partir de
# X_fuera_muestra. Ver tools/markdown/AUDITORIA_MIGRACION.md, hallazgo 2.5
df_hat_complex = df.copy()
df_hat_complex['Delta_x_hat'] = model_glm.predict(X_fuera_muestra)

processed.mkdir(parents=True, exist_ok=True)
df_hat_complex.to_parquet(processed / 'df_hat_complex.parquet', index=False)
print(f"predicciones generadas: {len(df_hat_complex):,} filas")

predicciones generadas: 36,851 filas
